# 04 — Densification diagnostic

**Why this exists.** A0 converges to ~635k primitives where vanilla SeaSplat
reaches 4,462,668 on the same scene — a 6× gap with no known cause. Three
hypotheses have been eliminated by static comparison:

| Eliminated | How |
|---|---|
| the rasterizer | identical forward *and* backward on identical inputs |
| densification constants | zero changed lines against the pinned baseline |
| upstream drift | `dxyang/seasplat` HEAD **is** `ddc6259` |

Everything comparable agrees and the outcomes still differ sixfold. So this
notebook stops diffing and watches both implementations densify, event by
event.

**It runs `train.py` directly, never the queue.** These are truncated runs and
must not be recorded as completed cells.

Run top to bottom. Roughly an hour: two 16,000-iteration runs, sequentially.


## 1. Drive and paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------------------
# The one place paths are defined. Everything else derives from DRIVE_ROOT.
#
#   e3dgsuw/
#     dataset/     the four scenes (original) + undistorted/  <- created below
#     dense/       M1 clouds, with SHA-256 sidecars
#     runs/        <cell>/<scene>/s<seed>/  -- one run, all of it together
#     analysis/    analyse.py output, figures, tables
#     run_ledger.json
# ---------------------------------------------------------------------------
DRIVE_ROOT   = '/content/drive/MyDrive/e3dgsuw'
DATASET_DIR  = f'{DRIVE_ROOT}/dataset'
DATA_UNDIST  = f'{DATASET_DIR}/undistorted'
DENSE_DIR    = f'{DRIVE_ROOT}/dense'
ANALYSIS_DIR = f'{DRIVE_ROOT}/analysis'

# Training reads from local disk, not Drive: the scene loader pulls every image
# at startup, and Drive's FUSE layer makes that far slower than a single copy.
LOCAL_DATA   = '/content/data'

REPO_URL  = 'https://github.com/dinanirham/An-Efficient-3D-Gaussian-Splatting-for-Underwater-3D-Reconstruction.git'
REPO_DIR  = '/content/e3dgsuw'
IMPL_DIR  = f'{REPO_DIR}/implementation'
SCENES    = ['Curasao', 'IUI3-RedSea', 'JapaneseGradens-RedSea', 'Panama']

import os
assert os.path.isdir(DRIVE_ROOT), (
    f'{DRIVE_ROOT} not found. Check the folder name, or edit DRIVE_ROOT above.')
for d in (DATA_UNDIST, DENSE_DIR, f'{DRIVE_ROOT}/runs', ANALYSIS_DIR):
    os.makedirs(d, exist_ok=True)

# Export them so the `!` cells below resolve "$DRIVE_ROOT" as a real shell
# variable. Relying on IPython to substitute notebook variables into magics
# works until it doesn't, and when it doesn't it substitutes nothing and the
# command runs against a silently truncated path rather than failing.
os.environ.update(
    DRIVE_ROOT=DRIVE_ROOT, DATASET_DIR=DATASET_DIR, DATA_UNDIST=DATA_UNDIST,
    DENSE_DIR=DENSE_DIR, ANALYSIS_DIR=ANALYSIS_DIR, LOCAL_DATA=LOCAL_DATA,
    REPO_DIR=REPO_DIR, IMPL_DIR=IMPL_DIR,
)


def find_originals():
    """Locate the four scenes under dataset/, however they were arranged.

    Accepts the scenes directly under dataset/, or nested one level (e.g.
    dataset/SeathruNeRF_dataset/). Returns the directory that contains them.
    """
    candidates = [DATASET_DIR] + [
        os.path.join(DATASET_DIR, d) for d in sorted(os.listdir(DATASET_DIR))
        if os.path.isdir(os.path.join(DATASET_DIR, d)) and d != 'undistorted'
    ]
    for base in candidates:
        if all(os.path.isdir(os.path.join(base, s)) for s in SCENES):
            return base
    return None


DATA_ORIG = find_originals()

# verify_undistort's T1 -- the check that would catch the undistortion gap --
# reads the *original* dataset. Point it at wherever it actually landed on
# Drive, or T1 reports "dataset not found" and the one check that matters here
# quietly stops testing anything.
if DATA_ORIG:
    os.environ['E3DGSUW_DATASET'] = DATA_ORIG

print('drive root :', DRIVE_ROOT)
print('originals  :', DATA_ORIG or 'NOT FOUND')
print('undistorted:', DATA_UNDIST)


## 2. GPU — must be an A100

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
cap  = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
print(f'torch {torch.__version__}  cuda {torch.version.cuda}  {name}  sm_{cap[0]}{cap[1]}')

# Every conclusion in this study is a between-cell contrast, and cells on
# different devices are not comparable. Stop now rather than produce a run
# that has to be discarded later.
assert 'A100' in name, f'Expected an A100, got {name!r}. Restart the runtime.'


## 3. Clone and build

In [ ]:
import os, subprocess

# Private repo? Add a Colab secret named GITHUB_TOKEN (key icon in the left
# sidebar) with a fine-grained read token, and toggle notebook access on.
# Read from Secrets rather than pasted into the cell: a pasted token is saved
# inside the .ipynb, which then travels wherever the notebook does.
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or None
    print('GITHUB_TOKEN: loaded from Colab Secrets')
except ImportError:
    pass                                  # not running under Colab
except Exception as e:                    # secret absent, or access not granted
    print(f'GITHUB_TOKEN: not available ({type(e).__name__}) -- '
          'fine for a public repo')

url = REPO_URL
if GITHUB_TOKEN:
    url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')

# Never let git fall back to an interactive credential prompt: in a notebook it
# hangs the cell indefinitely with nothing on screen to say why.
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}


def _redact(s):
    """Strip the token from git output -- git echoes the remote URL on failure,
    and notebook outputs are saved to the file and shared with it."""
    return s.replace(GITHUB_TOKEN, '***') if GITHUB_TOKEN else s


if os.path.isdir(REPO_DIR) and not os.path.isdir(f'{REPO_DIR}/.git'):
    raise RuntimeError(
        f'{REPO_DIR} exists but is not a git checkout -- probably a clone that '
        f'died partway. Delete it and re-run this cell.')

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git','clone','--depth','1',url,REPO_DIR],
                       capture_output=True, text=True, env=env)
    if r.returncode != 0:
        raise RuntimeError(
            'clone failed. If the repository is private, add a GITHUB_TOKEN '
            'secret in Colab and grant this notebook access.\n'
            f'{_redact(r.stderr)[-800:]}')
else:
    # Repoint the remote before pulling. The stored URL was written by an
    # earlier clone, which may have run without a token (or with a stale one);
    # injecting the token into `url` alone never reaches the pull.
    subprocess.run(['git','-C',REPO_DIR,'remote','set-url','origin',url],
                   check=True, env=env)
    r = subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],
                       capture_output=True, text=True, env=env)
    if r.returncode != 0:
        raise RuntimeError(
            'pull failed. If the repository is private, check the GITHUB_TOKEN '
            'secret is set and this notebook has access.\n'
            f'{_redact(r.stderr)[-800:]}')

# Fail here, naming the directory, rather than letting a later cell run from
# whatever the working directory happened to be.
assert os.path.isdir(IMPL_DIR), (
    f'clone produced no {IMPL_DIR}. Contents of {REPO_DIR}: '
    f'{sorted(os.listdir(REPO_DIR)) if os.path.isdir(REPO_DIR) else "missing"}')

os.chdir(IMPL_DIR)
print(subprocess.run(['git','-C',REPO_DIR,'log','--oneline','-1'],
                     capture_output=True, text=True).stdout.strip())
print('cwd:', os.getcwd())


In [ ]:
# Builds diff_gaussian_rasterization_ms and simple_knn against whatever torch
# Colab ships -- deliberately NOT installing our own, which would risk a
# mismatch between torch's CUDA and the toolkit the extensions compile with.
# Takes a few minutes; must be repeated each session.
#
# chdir explicitly rather than via `%cd $IMPL_DIR`: a magic whose variable fails
# to expand reports the *current* directory and continues, so the build then
# runs from the wrong place and fails two steps later with a bare
# "tools/setup_colab.sh: No such file or directory".
import os
assert os.path.isdir(IMPL_DIR), (
    f'{IMPL_DIR} not found -- run the "Clone the repository" cell above first.')
os.chdir(IMPL_DIR)
print('building in', os.getcwd())
!bash tools/setup_colab.sh


In [ ]:
import importlib, torch
for m in ('diff_gaussian_rasterization_ms', 'simple_knn'):
    importlib.import_module(m)
print('extensions import OK  |  torch', torch.__version__,
      '| cuda', torch.version.cuda)


## 4. Stage the dataset locally

**Not optional, and the step that has already been missed once.** Colab
recycles the VM, so `/content/data` is empty in a fresh session and both
trainings die at scene load — ours with a preflight message naming the missing
file, the reference with `Could not recognize scene type!`.


In [ ]:
import os, shutil, time
t0 = time.time()
os.makedirs(LOCAL_DATA, exist_ok=True)
for s in SCENES:
    src, dst = f'{DATA_UNDIST}/{s}', f'{LOCAL_DATA}/{s}'
    if not os.path.isdir(dst):
        shutil.copytree(src, dst)
print(f'staged in {time.time() - t0:.0f}s')

scene = f'{LOCAL_DATA}/Curasao'
for sub in ('images', 'sparse/0'):
    print(f'  {sub:<10} {os.path.isdir(f"{scene}/{sub}")}')
assert os.path.isfile(f'{scene}/sparse/0/cameras.bin'), 'staging incomplete'


## 5. Build and instrument the reference

`dxyang/seasplat` at its own HEAD, with its own rasterizer. The module names
differ (`diff_gaussian_rasterization` vs `..._ms`), so both coexist.

`instrument_reference` refuses to patch unless `densify_and_prune` matches the
expected upstream text exactly — so a mismatch here means the checkout is not
what we think it is, rather than a silently instrumented something-else.


In [ ]:
import os
if not os.path.isdir('/content/seasplat_ref'):
    !git clone -q --recursive https://github.com/dxyang/seasplat.git /content/seasplat_ref
    !pip install -q /content/seasplat_ref/submodules/diff-gaussian-rasterization
!cd /content/seasplat_ref && git log -1 --format="reference at %H  %ad" --date=short
!python -m tools.instrument_reference /content/seasplat_ref


## 6. Confirm the rasterizers still agree

Cheap, and it guards the comparison below: if the two forks ever stop matching
here, every event-level difference downstream is explained by that instead.


In [ ]:
!python -m tools.compare_rasterizers


## 7. Ours — 16,000 iterations

Past `densify_until_iter` (15000), so the whole densification phase is
captured. Scratch output directory; the ledger is untouched.

The `grep` deliberately has **no `^` anchor**: tqdm writes its progress bar
with `\r` and no trailing newline, so our line is appended to it and an
anchored pattern silently matches nothing.


In [ ]:
!cd "$IMPL_DIR" && python train.py \
  -s "$LOCAL_DATA"/Curasao --images images \
  --model_path /content/diag_ours --cell A0 --seed 2 \
  --iterations 16000 --test_iterations 16000 \
  --save_iterations 16000 --checkpoint_iterations 16000 \
  2>&1 | tee /content/diag_ours_full.txt \
       | grep --line-buffered -a "\[densify\]" | tee /content/diag_ours.txt


## 8. The reference — 16,000 iterations

Same scene, same seed, same schedule. Its `train.py` overwrites `model_path`
with `<source>/experiments/<date>/<exp>` regardless of `-m`, so its output
lands under the staged dataset — expected, and ephemeral.


In [ ]:
!cd /content/seasplat_ref && python train.py \
  -s "$LOCAL_DATA"/Curasao --images images --exp diag \
  --iterations 16000 --do_seathru --seathru_from_iter 10000 --eval --seed 2 \
  --test_iterations 16000 --save_iterations 16000 \
  --checkpoint_iterations 16000 \
  2>&1 | tee /content/diag_ref_full.txt \
       | grep --line-buffered -a "\[densify\]" | tee /content/diag_ref.txt


## 9. Compare

Aligns the two event streams — ours labels by iteration, the reference by
event index — and reports the first quantity to diverge. That names the
mechanism: `over_grad` means the signal differs, `clone`/`split` at equal
`over_grad` means the decision boundary differs, `prune` means pruning, with
the alpha/screen/world split saying which reason.


In [ ]:
!wc -l /content/diag_ours.txt /content/diag_ref.txt
!python -m tools.compare_densification /content/diag_ours.txt /content/diag_ref.txt


## 10. Keep the evidence

`/content` dies with the session. Copy to Drive so the diagnostic survives and
can be re-read without re-running an hour of training.


In [ ]:
import os, shutil
out = f'{DRIVE_ROOT}/analysis/densify_diagnostic'
os.makedirs(out, exist_ok=True)
for f in ('diag_ours.txt', 'diag_ref.txt',
          'diag_ours_full.txt', 'diag_ref_full.txt'):
    p = f'/content/{f}'
    if os.path.exists(p):
        shutil.copy(p, f'{out}/{f}')
        print(f'{f:<24} {os.path.getsize(p) / 1024:8.1f} KB')
print('\nsaved to', out)
